In [1]:
print("hello world")

hello world


In [1]:
import pandas as pd
df=pd.read_csv("100_Unique_QA_Dataset.csv")
df.sample(10)

,question,answer
0,What is the capital of France?,Paris
37,Who was the first person to step on the Moon?,Armstrong
6,What is the square root of 64?,8
11,Who developed the theory of relativity?,Albert-Einstein
38,What is the main ingredient in guacamole?,Avocado
70,What is the currency of Japan?,Yen
67,Which country has the pyramids of Giza?,Egypt
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
60,Which country is home to the Great Wall?,China
31,Which city is known as the Big Apple?,NewYork


In [6]:
#tokenize
def tokenize(text):
    text=text.lower()
    text=text.replace("?","")
    text=text.replace("'","")
    return text.split()

tokenize("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

In [16]:
#vocab
vocab={
    "<UNK>":0
}


In [17]:
def build_vocab(row):
    # print(row["question"],row["answer"])
    tokenize_que=tokenize(row["question"])
    tokenize_ans=tokenize(row["answer"])
    merged_token=tokenize_que+tokenize_ans
    # print(merged_token)
    for token in merged_token:
        if token not in vocab:
            vocab[token]=len(vocab)
df.apply(build_vocab,axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [18]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [19]:
#convert word to numerical indexces

def text_to_index(text,vocab):
    indexed_text=[]
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab["<UNK>"])
    return indexed_text


In [20]:
text_to_index("what is paris india",vocab)

[1, 2, 7, 73]

In [21]:
import torch 
from torch.utils.data import Dataset,DataLoader

In [24]:
class QAdataset(Dataset):
    def __init__(self, df, vocab):
        self.df=df
        self.vocab=vocab

    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        num_que=text_to_index(self.df.iloc[index]["question"],self.vocab)
        num_ans=text_to_index(self.df.iloc[index]["answer"],self.vocab)
        return torch.tensor(num_que),torch.tensor(num_ans)

In [25]:
dataset=QAdataset(df,vocab)

In [30]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [31]:
import torch.nn as nn


In [43]:
class SimpleRNN(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn=nn.RNN(50,64,batch_first=True)
        self.fl=nn.Linear(64,vocab_size)
    
    def forward(self,question):
        embedded_que=self.embedding(question)
        hidden,final=self.rnn(embedded_que)
        output=self.fl(final.squeeze(0))
        return output

In [33]:
learning_rate=0.001
epochs=20

In [45]:
model=SimpleRNN(len(vocab))

In [46]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [57]:
for epoch in range(epochs):
    total_loss=0
    for question,answer in dataloader:
        optimizer.zero_grad()
        # forward pass
        ouptut=model(question)
        # loss
        loss= criterion(ouptut,answer[0])

        # calculate grads 
        loss.backward()
        #update

        optimizer.step()

        total_loss=total_loss+loss.item()
    print(f"epoch: {epoch}, LOSS: {total_loss:4f}")

epoch: 0, LOSS: 0.000847
epoch: 1, LOSS: 0.000810
epoch: 2, LOSS: 0.000775
epoch: 3, LOSS: 0.000742
epoch: 4, LOSS: 0.000710
epoch: 5, LOSS: 0.000680
epoch: 6, LOSS: 0.000650
epoch: 7, LOSS: 0.000622
epoch: 8, LOSS: 0.000593
epoch: 9, LOSS: 0.000569
epoch: 10, LOSS: 0.000543
epoch: 11, LOSS: 0.000520
epoch: 12, LOSS: 0.000497
epoch: 13, LOSS: 0.000478
epoch: 14, LOSS: 0.000455
epoch: 15, LOSS: 0.000435
epoch: 16, LOSS: 0.000417
epoch: 17, LOSS: 0.000399
epoch: 18, LOSS: 0.000380
epoch: 19, LOSS: 0.000363


In [71]:
def predict(model,question,threshold=0.5):
    #convert question to number
    numerical_question=text_to_index(question,vocab)

    #tensor
    question_tensor=torch.tensor(numerical_question).unsqueeze(0)
    # print(question_tensor.shape)

    # send to model

    ouptut=model(question_tensor)
    # convert logits to prob
    probs=torch.nn.functional.softmax(ouptut,dim=1)

    #find index of max probablity
    value,index=torch.max(probs,dim=1)
    print(value,index)

    if value<threshold:
        print("i don't know")
    else:
        print(list(vocab.keys())[index])

predict(model," capital of france")


tensor([1.0000], grad_fn=<MaxBackward0>) tensor([7])
paris
